In [3]:
import re
import time
import numpy as np
import pandas as pd
import requests
import trafilatura
from datetime import datetime, timedelta
from collections import defaultdict

In [4]:



FINANCE_KEYWORDS = {
    "inflation", "cpi", "core inflation", "price pressures", "prices",
    "interest rate", "interest rates", "rate hike", "rate cut",
    "bank of england", "boe", "federal reserve", "fed", "ecb",
    "monetary policy", "tightening", "easing", "hawkish", "dovish",
    "wages", "labour market", "labor market", "consumer prices", "central bank"
}

SENTENCE_KEYWORDS = {
    "inflation", "cpi", "core inflation", "interest rate", "interest rates",
    "rate hike", "rate cut", "bank of england", "boe", "federal reserve", "fed",
    "ecb", "central bank", "monetary policy", "tightening", "easing",
    "hawkish", "dovish", "wages", "prices", "consumer prices",
    "sticky inflation", "services inflation", "disinflation"
}

FORWARD_LOOKING_KEYWORDS = {
    "expect", "expected", "expects", "forecast", "forecasts", "projected",
    "outlook", "likely", "unlikely", "could", "may", "might", "should",
    "anticipated", "anticipate", "signals", "signalled", "guidance"
}


def fetch_newsapi_articles(
    api_key,
    query,
    from_date=None,
    to_date=None,
    page_size=50,
    pages=1,
    language="en",
    search_in="title,description",
    domains=None,
    exclude_domains=None,
    sort_by="publishedAt"
):
    """
    Fetch article metadata from NewsAPI /v2/everything.
    This only gets metadata, not full article text.
    """
    url = "https://newsapi.org/v2/everything"
    all_articles = []

    for page in range(1, pages + 1):
        params = {
            "q": query,
            "language": language,
            "pageSize": page_size,
            "page": page,
            "sortBy": sort_by,
            "searchIn": search_in,
            "apiKey": api_key,
        }

        if from_date:
            params["from"] = from_date
        if to_date:
            params["to"] = to_date
        if domains:
            params["domains"] = ",".join(domains)
        if exclude_domains:
            params["excludeDomains"] = ",".join(exclude_domains)

        r = requests.get(url, params=params, timeout=20)
        r.raise_for_status()
        data = r.json()

        articles = data.get("articles", [])
        all_articles.extend(articles)

        if len(articles) < page_size:
            break

        time.sleep(0.5)

    return all_articles


def is_financial_or_inflation_related(title, description=""):
    text = f"{title or ''} {description or ''}".lower()
    return any(keyword in text for keyword in FINANCE_KEYWORDS)


def extract_article_text(url):
    """
    Download and extract main article text using Trafilatura.
    Returns None if extraction fails.
    """
    downloaded = trafilatura.fetch_url(url)
    if not downloaded:
        return None

    text = trafilatura.extract(
        downloaded,
        favor_precision=True,
        include_comments=False
    )
    return text


def split_into_sentences(text):
    if not text:
        return []
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]


def sentence_score(sentence, title=""):
    """
    Score a sentence for relevance.
    Higher score = more likely useful for inflation / policy sentiment analysis.
    """
    s_lower = sentence.lower()
    title_lower = (title or "").lower()

    # basic keyword hits
    keyword_score = sum(kw in s_lower for kw in SENTENCE_KEYWORDS)

    # forward-looking language is often more useful for market sentiment
    forward_score = sum(kw in s_lower for kw in FORWARD_LOOKING_KEYWORDS)

    # overlap with title words
    title_words = set(re.findall(r"\b[a-z]{4,}\b", title_lower))
    sent_words = set(re.findall(r"\b[a-z]{4,}\b", s_lower))
    overlap_score = len(title_words & sent_words)

    # prefer medium-length sentences over tiny useless fragments
    word_count = len(sentence.split())
    if 8 <= word_count <= 40:
        length_score = 2
    elif 5 <= word_count <= 60:
        length_score = 1
    else:
        length_score = 0

    return (
        3 * keyword_score
        + 2 * forward_score
        + overlap_score
        + length_score
    )


def extract_key_sentences(text, title="", max_sentences=5, min_score=2):
    """
    Extract top-N relevant sentences from article text.
    """
    sentences = split_into_sentences(text)
    scored = []

    for i, s in enumerate(sentences):
        score = sentence_score(s, title=title)

        # slight preference for earlier sentences because journalists
        # keep pretending the important bit belongs near the top
        position_bonus = max(0, 2 - i * 0.1)
        total_score = score + position_bonus

        if total_score >= min_score:
            scored.append((total_score, s))

    scored.sort(key=lambda x: x[0], reverse=True)

    chosen = []
    seen = set()
    for _, s in scored:
        if s not in seen:
            chosen.append(s)
            seen.add(s)
        if len(chosen) >= max_sentences:
            break

    return chosen


def parse_article_day(article):
    published = article.get("publishedAt")
    if not published:
        return None
    # publishedAt usually looks like 2026-04-03T12:34:56Z
    return published[:10]


def default_sentiment_function(text):
    """
    Placeholder.
    Replace this with your trained model function.
    Should return a scalar score, e.g.:
      negative = -1
      neutral  = 0
      positive = +1
    or a continuous score.
    """
    return 0.0


def build_daily_news_results(
    api_key,
    query,
    start_date,
    end_date,
    sentiment_fn=None,
    max_pages_per_day=1,
    page_size=50,
    max_sentences_per_article=5,
    domains=None,
    exclude_domains=None
):
    """
    Build results[day] dictionary:
      results[day]["articles"] -> list of processed article dicts
      results[day]["daily_sentiment_mean"] -> average sentiment of article snippets
      results[day]["n_articles"] -> number of usable articles
    """
    if sentiment_fn is None:
        sentiment_fn = default_sentiment_function

    results = defaultdict(lambda: {
        "articles": [],
        "daily_sentiment_mean": None,
        "n_articles": 0
    })

    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date, "%Y-%m-%d")

    current = start_dt
    while current <= end_dt:
        day_str = current.strftime("%Y-%m-%d")
        next_day_str = (current + timedelta(days=1)).strftime("%Y-%m-%d")

        # Fetch one day at a time to control quota and keep clean time series
        raw_articles = fetch_newsapi_articles(
            api_key=api_key,
            query=query,
            from_date=day_str,
            to_date=next_day_str,
            page_size=page_size,
            pages=max_pages_per_day,
            language="en",
            search_in="title,description",
            domains=domains,
            exclude_domains=exclude_domains,
            sort_by="publishedAt"
        )

        # Cheap metadata filter before full-text extraction
        filtered_articles = []
        seen_urls = set()

        for article in raw_articles:
            url = article.get("url")
            if not url or url in seen_urls:
                continue
            seen_urls.add(url)

            title = article.get("title", "")
            desc = article.get("description", "")

            if is_financial_or_inflation_related(title, desc):
                filtered_articles.append(article)

        day_sentiments = []

        for article in filtered_articles:
            title = article.get("title", "")
            url = article.get("url", "")
            source = (article.get("source") or {}).get("name")
            published_at = article.get("publishedAt")

            text = extract_article_text(url)
            if not text:
                continue

            key_sentences = extract_key_sentences(
                text,
                title=title,
                max_sentences=max_sentences_per_article
            )

            if not key_sentences:
                continue

            snippet_for_model = " ".join(key_sentences)
            sentiment_score = sentiment_fn(snippet_for_model)

            results[day_str]["articles"].append({
                "title": title,
                "url": url,
                "source": source,
                "publishedAt": published_at,
                "key_sentences": key_sentences,
                "snippet_for_model": snippet_for_model,
                "sentiment_score": sentiment_score,
            })

            day_sentiments.append(sentiment_score)

        results[day_str]["n_articles"] = len(results[day_str]["articles"])
        if day_sentiments:
            results[day_str]["daily_sentiment_mean"] = sum(day_sentiments) / len(day_sentiments)

        current += timedelta(days=1)

    return dict(results)

In [5]:
NEWSAPI_KEY = "0cdaaa7d6b5c49d69867ddafa4fb4229"

QUERY = (
    '("inflation" OR "CPI" OR "core inflation" OR "interest rates" '
    'OR "rate hike" OR "rate cut" OR "monetary policy" '
    'OR "central bank" OR "Bank of England" OR "Federal Reserve" OR "ECB")'
)

TRUSTED_DOMAINS = [
    "reuters.com",
    "ft.com",
    "bloomberg.com",
    "wsj.com",
    "cnbc.com"
]

def my_sentiment_fn(text):
    # Replace this with your real classifier
    # Example:
    # return predict_sentiment(text)
    return 0.0

results = build_daily_news_results(
    api_key=NEWSAPI_KEY,
    query=QUERY,
    start_date="2026-03-20",
    end_date="2026-04-03",
    sentiment_fn=my_sentiment_fn,
    max_pages_per_day=1,
    page_size=20,
    max_sentences_per_article=5,
    domains=TRUSTED_DOMAINS
)



In [8]:
import pandas as pd 

results_df = pd.DataFrame(results)
results_df.head()

,2026-03-20,2026-03-21,2026-03-22,2026-03-23,2026-03-24,2026-03-25,2026-03-26,2026-03-27,2026-03-28,2026-03-29,2026-03-30,2026-03-31,2026-04-01,2026-04-02,2026-04-03
articles,[{'title': 'Analysis: Trump's unshackled presi...,[{'title': 'The price of menstrual products is...,[{'title': 'Japan core inflation in February m...,[{'title': 'Stagflation alarm bells ring in th...,[{'title': 'ECB ready to hike rates even if ex...,[{'title': 'Sen. Warren rips Federal Reserve c...,[{'title': 'Analysis: What might trip up Kevin...,[{'title': 'Worried about Strait of Hormuz inf...,[{'title': 'Global week ahead: Why emergency G...,[{'title': 'Powell sees inflation outlook in c...,"[{'title': 'Wall Street's rough month, Powell'...",[{'title': 'Prices may rise more this year tha...,[{'title': 'Cramer’s week ahead: Two key econo...,[{'title': 'Cramer’s week ahead: Two key econo...,[]
daily_sentiment_mean,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None
n_articles,5,2,4,4,4,6,5,3,2,4,6,4,2,1,0
